# Task 1 — Gemini và OpenAI trên cùng pipeline

Notebook dùng split cố định **26 train / 6 validation**, cùng prompt, rule và metric cho hai provider. Preflight không gọi API; các cell chạy thật cần bật `RUN_LIVE = True`. Kết quả cũ của Gemini trên split 13/19 chỉ là lịch sử.

In [4]:
%pip -q install "pyyaml>=6" "openai>=2,<3" "pydantic>=2.5" gdown

In [6]:
from google.colab import drive
from pathlib import Path
import os, sys, zipfile, subprocess, json, shutil

drive.mount('/content/drive')
PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_DIR = PROJECT_HOME / 'llm-final-project'
DRIVE_DATA = PROJECT_HOME / 'data'
RAW_DIR = DRIVE_DATA / 'raw'
SPLITS_DIR = DRIVE_DATA / 'splits'
OUTPUT_DIR = DRIVE_DATA / 'outputs' / 'task1'
RUNNER = REPO_DIR / 'scripts' / 'run_task1.py'
if not RUNNER.is_file():
    raise FileNotFoundError(f'Đồng bộ branch task-1 có runner chung lên Drive: {RUNNER}')
sys.path.insert(0, str(REPO_DIR))
from src.data.loader import resolve_dataset_root
try:
    DATA_DIR = resolve_dataset_root(RAW_DIR)
except FileNotFoundError:
    import gdown
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    archive = Path('/content/task1_sample_dataset.zip')
    gdown.download(id='1iqkw6ZWUV6-I6rsVQdqHBOhiRgRkk_UZ', output=str(archive), quiet=False)
    if not zipfile.is_zipfile(archive):
        raise RuntimeError('Tệp tải về không phải ZIP. Hãy đặt dataset vào Drive data/raw.')
    with zipfile.ZipFile(archive) as zf:
        root = RAW_DIR.resolve()
        if any(not (root / name).resolve().is_relative_to(root) for name in zf.namelist()):
            raise ValueError('ZIP chứa đường dẫn không an toàn')
        zf.extractall(RAW_DIR)
    DATA_DIR = resolve_dataset_root(RAW_DIR)
# Use the committed 26/6 split on Drive; archive a prior 13/19 split first.
canonical = REPO_DIR / 'data' / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for part in ('train', 'val'):
    source = canonical / f'{part}_ids.json'
    target = SPLITS_DIR / f'{part}_ids.json'
    if not source.is_file():
        raise FileNotFoundError(f'Missing canonical split: {source}')
    if target.is_file() and json.loads(target.read_text(encoding='utf-8')) != json.loads(source.read_text(encoding='utf-8')):
        old = SPLITS_DIR / 'history_before_26_6'
        old.mkdir(exist_ok=True)
        shutil.copy2(target, old / target.name)
    if not target.is_file() or json.loads(target.read_text(encoding='utf-8')) != json.loads(source.read_text(encoding='utf-8')):
        shutil.copy2(source, target)
print('Dataset:', DATA_DIR)
print('Split:', SPLITS_DIR)
print('Output:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Đồng bộ branch task-1 có runner chung lên Drive: /content/drive/MyDrive/LLM/Final Project/llm-final-project/scripts/run_task1.py

In [ ]:
def task1(provider, *flags):
    command = [sys.executable, str(RUNNER), '--provider', provider,
               '--data-dir', str(DATA_DIR), '--splits-dir', str(SPLITS_DIR),
               '--output-dir', str(OUTPUT_DIR), *flags]
    subprocess.run(command, cwd=REPO_DIR, check=True)

for provider in ('gemini', 'openai'):
    task1(provider, '--preflight')

## Chạy API

Thêm `GEMINI_API_KEY` và `OPENAI_API_KEY` vào Colab Secrets, bật quyền truy cập notebook. Bật `RUN_LIVE` khi muốn chạy. Chạy smoke test trước; kết quả nằm trong thư mục riêng theo provider và model. Hai lượt chạy đầy đủ đều dùng 6 validation.

In [ ]:
RUN_LIVE = False  # Đổi thành True để cho phép gọi API
if RUN_LIVE:
    from google.colab import userdata
    for key in ('GEMINI_API_KEY', 'OPENAI_API_KEY'):
        value = userdata.get(key)
        if not value:
            raise RuntimeError(f'Thiếu {key} trong Colab Secrets')
        os.environ[key] = value
        del value
    print('Đã nạp khóa từ Colab Secrets; không in giá trị.')

In [ ]:
if RUN_LIVE:
    for provider in ('gemini', 'openai'):
        task1(provider, '--smoke')

In [ ]:
if RUN_LIVE:
    for provider in ('gemini', 'openai'):
        task1(provider)
    print('Kết quả:', OUTPUT_DIR)
else:
    print('Preflight hoàn tất. Bật RUN_LIVE để chạy API.')